# Check-in 2: baselines and imprint forgery for Ring-ID, SFW, WIND and PRC

**CMPE 261, Group 6: Forging the Watermark.** This notebook is the actual Check-in 2 run, with its outputs kept as executed.

Check-in 1 tested two semantic watermarks (Tree-Ring and Gaussian Shading). This run carries the same pipeline to the other four core schemes, with nothing else changed:

| Setting | Value (same as Check-in 1) |
|---|---|
| Target model | Stable Diffusion 2.1-base, 50 DDIM steps, guidance 7.5, 512×512 |
| Attack | Imprint forgery (Müller et al., CVPR 2025), same-model attacker (condition C1) |
| Cover photos | the same 5 COCO val2017 photographs |
| Detector check | every 10 imprint steps; stop at the first detection |

It runs in two stages:

1. **F1, baselines.** Does each detector work before any attack? 50 watermarked + 50 unwatermarked generations + 50 real COCO photos per scheme. Inclusion rule: TPR ≥ 90% and FPR ≤ 5%.
2. **F2, forgery.** Can a real photo be pushed until the detector calls it watermarked? 5 photos per scheme, up to 150 steps.

**Hardware:** Colab A100 40 GB. **Results:** written to Google Drive (`OUT`). Every stage appends to `runs.csv` and resumes from it after a disconnect.

## 1. Settings and GPU check

`OUT` points to Google Drive, so results survive a Colab disconnect. The printout confirms the GPU (an A100 for this run).

In [1]:
# Use the latest team code and save results to Google Drive.
BRANCH = "main"
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cmpe261-watermark-forgery/checkin2"   # Check-in 2 results folder
else:
    OUT = "/content/cmpe261-watermark-forgery/results/checkin2"
# Show which GPU Colab assigned (T4 is ~8x slower than A100 for this workload).
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Get the code

Fresh clone of the team repository on `main`. Deleting any old copy first avoids a stale checkout of the `checkin-1` branch.

In [2]:
# Remove any earlier copy and clone the repository fresh.
%cd /content
!rm -rf /content/cmpe261-watermark-forgery
!git clone https://github.com/yashashav-dk/cmpe261-watermark-forgery.git

/content
Cloning into 'cmpe261-watermark-forgery'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 157 (delta 58), reused 153 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 6.52 MiB | 12.41 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [3]:
# Confirm we are on the main branch.
%cd /content/cmpe261-watermark-forgery
!git branch

/content/cmpe261-watermark-forgery
* main


## 3. Install dependencies

Installs our package (`wmforge`) and MarkDiffusion pinned to a fixed commit, so the watermarking schemes and detectors are identical across runs. `cuda True` on the last line means the GPU is usable. The red `cuda-python` resolver message is harmless.

In [4]:
import os
%cd /content
if not os.path.isdir("cmpe261-watermark-forgery"):
    !git clone --branch {BRANCH} https://github.com/yashashav-dk/cmpe261-watermark-forgery.git
%cd /content/cmpe261-watermark-forgery
!git pull --ff-only
# Our package plus MarkDiffusion at a pinned commit (same version as Check-in 1).
!pip install -q -e ".[attack]" "markdiffusion @ git+https://github.com/THU-BPM/MarkDiffusion@9d81656d1a5f9e5194fc2f727bb795ef29e53809"
# Colab's preinstalled torchaudio is built for a newer torch than MarkDiffusion's pin allows; nothing here uses it.
!pip uninstall -y -q torchaudio
!python -c "import torch, diffusers, markdiffusion; print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| cuda', torch.cuda.is_available())"

/content
/content/cmpe261-watermark-forgery
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Extra packages for SFW and PRC (found during this run)

MarkDiffusion does not install these, and without them the two schemes crash on import:

| Package | Needed by | Error without it |
|---|---|---|
| `qrcode` | SFW | `No module named 'qrcode'` |
| `galois` | PRC | `No module named 'galois'` |
| `ldpc<2` | PRC | `No module named 'ldpc'`; version 2 renamed the `bp_decoder` PRC imports |

Ring-ID and WIND need nothing extra. Check-in 1's schemes (Tree-Ring, Gaussian Shading) never hit this.

In [5]:
# Missing dependencies discovered in the dry run: qrcode (SFW), galois and ldpc<2 (PRC).
!pip install -q qrcode galois "ldpc<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.4 MB/s eta 0:00:00


## 4. Dry run

Two prompts and two photos per scheme, about 3 minutes in total. It checks that every scheme loads, generates and detects before spending an hour of GPU time. All four print `finished … clean, real` and pass 2/2 (wide intervals at n = 2 are expected).

In [ ]:
# Quick end-to-end check: 2 clean prompts and 2 real photos per scheme, written to a throwaway folder.
!python -m wmforge.smoke --out results/dryrun_f1 --schemes RI SFW WIND PRC --stages clean real --n-clean 2 --n-real 2
!python -m wmforge.report --run results/dryrun_f1

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
scheduler_config.json: 100% 346/346 [00:00<00:00, 1.55MB/s]
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
model_index.json: 100% 543/543 [00:00<00:00, 2.03MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/342 [00:00<?, ?B/s]           

Fetching 12 files:  17% 2/12 [00:00<00:01,  7.70it/s]
Reconstructing (incomplete total...):   0% 342/525k [00:00<06:36, 1.32kB/s]
Reconstructing (incomplete 

## 5. F1: clean and real-photo baselines

For each scheme: 50 prompts generated with and without the watermark, plus 50 real COCO photographs, each run through the detector. Each `100% 50/50` bar is one image (50 sampling or inversion steps, about 3.5 s on the A100). About 45 minutes in total.

The long-prompt `truncated because CLIP can only handle 77 tokens` messages are expected. A few DiffusionDB prompts exceed CLIP's limit, as noted in Check-in 1's data exploration.

In [ ]:
# F1 baselines for the four new schemes: clean (watermarked vs unwatermarked) and real photos.
# Rows are appended to OUT/runs.csv; rerunning skips anything already recorded.
!python -m wmforge.smoke --out {OUT} --schemes RI SFW WIND PRC --stages clean real

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]
Loading weights: 100% 372/372 [00:00<00:00, 4883.97it/s]
Loading pipeline components...: 100% 6/6 [00:00<00:00,  8.62it/s]
100% 50/50 [00:03<00:00, 14.41it/s]
100% 50/50 [00:03<00:00, 14.49it/s]
100% 50/50 [00:03<00:00, 14.48it/s]
100% 50/50 [00:03<00:00, 14.51it/s]
100% 50/50 [00:03<00:00, 14.47it/s]
100% 50/50 [00:03<00:00, 14.51it/s]
100% 50/50 [00:03<00:00, 14.50it/s]
100% 50/50 [00:03<00:00, 14.50it/s]
100% 50/5

## 6. F2: imprint forgery

For each scheme and each of the 5 cover photos:

1. Generate one watermarked **reference** image.
2. Optimize the real **cover** photo so that its DDIM-inverted noise matches the reference's.
3. Run the detector every 10 steps. Stop as soon as it says "watermarked", or at 150 steps.

The progress bars here are the reference generations and the detector checks (DDIM inversion). The optimization steps themselves do not print. About 8.4 s per imprint step on the A100.

In [6]:
# F2 imprint forgery: 5 real cover photos per scheme, cap of 150 imprint steps.
# Saves cover / reference / forged images under OUT/images and appends results to OUT/runs.csv.
!python -m wmforge.smoke --out {OUT} --schemes RI SFW WIND PRC --stages imprint --n-covers 5 --max-steps 150

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
scheduler_config.json: 100% 346/346 [00:00<00:00, 1.98MB/s]
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
model_index.json: 100% 543/543 [00:00<00:00, 3.35MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0% 0/12 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/342 [00:00<?, ?B/s]           

Fetching 12 files:  17% 2/12 [00:00<00:00, 18.05it/s]
Reconstructing (incomplete total...):   0% 342/1.36G [00:00<121:57:39, 3.10kB/s]
Reconstructing (incomp

## 7. Results

Computed from `runs.csv` (committed as `results/checkin2/runs.csv`). Figures: `results/checkin2/figures/`, drawn by `scripts/checkin2_figures.py`, which needs no GPU.

**F1 baselines.** All four pass the inclusion rule.

| Scheme | TPR, watermarked | FPR, unwatermarked | FPR, real photos |
|---|---|---|---|
| Ring-ID | 49/50 | 0/50 | 0/50 |
| SFW | 49/50 | 0/50 | 0/50 |
| WIND | 50/50 | 0/50 | 0/50 |
| PRC | 50/50 | 0/50 | 0/50 |

**F2 forgery.** Every real photo was forged on every new scheme.

| Scheme | Forged | Steps to first detection (median, range) | PSNR of forgery (median, range) |
|---|---|---|---|
| PRC | 5/5 | 20 (20–30) | 27.7 dB (22.0–30.5) |
| WIND | 5/5 | 30 (30–50) | 27.0 dB (21.2–29.4) |
| Ring-ID | 5/5 | 70 (50–120) | 26.0 dB (20.1–27.9) |
| SFW | 5/5 | 90 (80–140) | 25.1 dB (20.0–27.3) |

With Check-in 1, the order from easiest to hardest to forge is: Gaussian Shading (10 steps, 5/5) → PRC → WIND → Ring-ID → SFW → Tree-Ring (4/5 within 150).

**Caveats.** Five covers give wide confidence intervals: 5/5 is 57%–100% at 95%. Ring-ID and SFW forgeries crossed the threshold of 50 only narrowly (scores 49.0–49.9). PRC is reported by verdict only, because its score direction is unverified. The same photo (COCO 32887) was the hardest to forge for every scheme and ended near 20 dB, a visible cost.